# hERG ChEMBL post-processing

This notebook implements the post-query cleaning workflow:
- standardize concentration endpoints to nM
- convert IC50/Ki to p-scale (`-log10(M)`) where possible
- preserve and model censoring/inequalities
- infer pIC50-like values for percent inhibition rows using single-point assumptions
- remove or flag missing/invalid structures
- resolve duplicate compound-assay entries with prioritized selection + median aggregation
- write cleaned outputs and `filtering.log`


In [2]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

RAW_PATH = Path("data/herg_raw.csv")
CLEAN_OUTPUT_PATH = Path("data/herg_postprocessed.csv")
PERCENT_OUTPUT_PATH = Path("data/herg_percent_inhibition.csv")
LOG_PATH = Path("filtering.log")

UNIT_TO_NM = {
    "m": 1e9,
    "mm": 1e6,
    "um": 1e3,
    "nm": 1.0,
    "pm": 1e-3,
    "fm": 1e-6,
}
UNIT_TO_M = {
    "m": 1.0,
    "mm": 1e-3,
    "um": 1e-6,
    "nm": 1e-9,
    "pm": 1e-12,
    "fm": 1e-15,
}
MASS_PER_VOL_TO_G_PER_L = {
    "ug/ml": 1e-3,
    "mg/ml": 1.0,
}
ALLOWED_RELATIONS = {"=", ">", "<", ">=", "<=", "~"}
RELATION_INVERT_FOR_P = {
    "=": "=",
    "~": "~",
    ">": "<",
    ">=": "<=",
    "<": ">",
    "<=": ">=",
}
GROUP_COLS = ["molecule_chembl_id", "assay_chembl_id"]

PSEUDO_PERCENT_MIN = 0.1
PSEUDO_PERCENT_MAX = 99.9
HILL_N_ASSUMED = 1.0

MOLAR_PATTERN = re.compile(r"(?i)(\d+(?:\.\d+)?)\s*(nm|u[m]|mm|m|pm|fm)\b")
MASS_PATTERN = re.compile(r"(?i)(\d+(?:\.\d+)?)\s*(ug\s*/\s*ml|mg\s*/\s*ml)\b")


def clean_text(value):
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if isinstance(value, str):
        text = " ".join(value.split())
        return text if text else None
    return str(value)


def normalize_unit(unit_value):
    text = clean_text(unit_value)
    if text is None:
        return None
    return (
        text.lower()
        .replace(chr(181), "u")
        .replace(chr(956), "u")
        .replace(" ", "")
    )


def normalize_relation(rel):
    text = clean_text(rel)
    return text if text in ALLOWED_RELATIONS else None


def to_nm(value_num, unit_norm):
    if pd.isna(value_num) or unit_norm not in UNIT_TO_NM:
        return np.nan
    return float(value_num) * UNIT_TO_NM[unit_norm]


def classify_endpoint(standard_type, standard_units):
    st = (clean_text(standard_type) or "").upper()
    su = (clean_text(standard_units) or "").strip()
    if su == "%" or "INHIB" in st or "PERCENT" in st or st in {"INH", "INHIBITION", "% OF CONTROL"}:
        return "percent_inhibition"
    if st in {"IC50", "KI"}:
        return "ic50_ki"
    if st in {"AC50", "EC50", "IC20", "IC25", "EC10", "LOG IC50", "INFLECTION POINT"}:
        return "other_concentration"
    if normalize_unit(standard_units) in UNIT_TO_NM:
        return "other_concentration"
    return "other_nonstandard"


def censoring_type(relation):
    if relation in {">", ">="}:
        return "right_censored_lower_bound"
    if relation in {"<", "<="}:
        return "left_censored_upper_bound"
    return "uncensored_or_approx"


def first_non_null(series):
    for value in series:
        if pd.notna(value):
            return value
    return np.nan


def smiles_validator():
    try:
        from rdkit import Chem

        def validate(smiles):
            text = clean_text(smiles)
            return bool(text) and Chem.MolFromSmiles(text) is not None

        return validate, "rdkit"
    except Exception:
        pattern = re.compile(r"^[A-Za-z0-9@+\-\[\]\(\)=#$\\\/%\.:\*]+$")

        def validate(smiles):
            text = clean_text(smiles)
            return bool(text) and bool(pattern.fullmatch(text))

        return validate, "regex_fallback_no_rdkit"


def numeric_or_nan(value):
    try:
        return float(value)
    except Exception:
        return np.nan


def parse_test_concentration(text, molecular_weight):
    text_norm = clean_text(text)
    if text_norm is None:
        return {
            "test_concentration_M": np.nan,
            "test_concentration_unit": None,
            "test_concentration_raw_value": np.nan,
            "test_concentration_method": None,
        }

    text_norm = text_norm.replace(chr(181), "u").replace(chr(956), "u")

    molar_match = MOLAR_PATTERN.search(text_norm)
    if molar_match:
        value = float(molar_match.group(1))
        unit = molar_match.group(2).lower()
        return {
            "test_concentration_M": value * UNIT_TO_M[unit],
            "test_concentration_unit": unit,
            "test_concentration_raw_value": value,
            "test_concentration_method": "parsed_molar_text",
        }

    mass_match = MASS_PATTERN.search(text_norm)
    if mass_match:
        value = float(mass_match.group(1))
        unit = mass_match.group(2).lower().replace(" ", "")
        if not pd.isna(molecular_weight) and molecular_weight > 0:
            g_per_l = value * MASS_PER_VOL_TO_G_PER_L[unit]
            molar = g_per_l / molecular_weight
            return {
                "test_concentration_M": molar,
                "test_concentration_unit": unit,
                "test_concentration_raw_value": value,
                "test_concentration_method": "parsed_mass_text_with_mw",
            }
        return {
            "test_concentration_M": np.nan,
            "test_concentration_unit": unit,
            "test_concentration_raw_value": value,
            "test_concentration_method": "parsed_mass_text_missing_mw",
        }

    return {
        "test_concentration_M": np.nan,
        "test_concentration_unit": None,
        "test_concentration_raw_value": np.nan,
        "test_concentration_method": None,
    }


def infer_test_concentration(row):
    molecular_weight = np.nan
    for mw_col in ("full_mwt", "molecular_weight", "mw_freebase"):
        if mw_col in row.index:
            molecular_weight = numeric_or_nan(row.get(mw_col))
            if not pd.isna(molecular_weight):
                break

    for source_col in ("assay_description", "activity_comment"):
        parsed = parse_test_concentration(row.get(source_col), molecular_weight)
        method = parsed["test_concentration_method"]
        if method is not None:
            parsed["test_concentration_source"] = source_col
            return parsed

    parsed = parse_test_concentration(None, molecular_weight)
    parsed["test_concentration_source"] = None
    return parsed


df_raw = pd.read_csv(RAW_PATH, low_memory=False)
df = df_raw.copy()

for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].map(clean_text)

df["standard_value_num"] = pd.to_numeric(df["standard_value"], errors="coerce")
df["pchembl_value_num"] = pd.to_numeric(
    df.get("pchembl_value_num", df["pchembl_value"]), errors="coerce"
)
df["document_year"] = pd.to_numeric(df["document_year"], errors="coerce").astype("Int64")

df["standard_units_norm"] = df["standard_units"].map(normalize_unit)
df["relation_preserved"] = df["standard_relation"].map(normalize_relation)
df["endpoint_class"] = [
    classify_endpoint(st, su) for st, su in zip(df["standard_type"], df["standard_units"])
]

df["standard_value_nM"] = [to_nm(v, u) for v, u in zip(df["standard_value_num"], df["standard_units_norm"])]
df["standard_value_M"] = df["standard_value_nM"] * 1e-9
positive_molar = df["standard_value_M"] > 0
df["p_activity_bound"] = np.where(positive_molar, -np.log10(df["standard_value_M"]), np.nan)
df["p_activity_relation"] = df["relation_preserved"].map(RELATION_INVERT_FOR_P)
df["censoring_type"] = df["relation_preserved"].map(censoring_type)

std_type_norm = df["standard_type"].fillna("").str.upper()
ic50_ki_mask = std_type_norm.isin({"IC50", "KI"}) & df["p_activity_bound"].notna()
df["p_ic50_ki"] = np.where(ic50_ki_mask, df["p_activity_bound"], np.nan)
df["p_ic50_ki_relation"] = np.where(ic50_ki_mask, df["p_activity_relation"], None)

df["is_percent_inhibition"] = df["endpoint_class"].eq("percent_inhibition")
parsed_conc_df = df.apply(infer_test_concentration, axis=1, result_type="expand")
df = pd.concat([df, parsed_conc_df], axis=1)
df["percent_inhibition_value"] = np.where(df["is_percent_inhibition"], df["standard_value_num"], np.nan)

valid_percent = df["percent_inhibition_value"].between(
    PSEUDO_PERCENT_MIN, PSEUDO_PERCENT_MAX, inclusive="both"
)
valid_single_point = (
    df["is_percent_inhibition"]
    & df["test_concentration_M"].notna()
    & (df["test_concentration_M"] > 0)
    & valid_percent
)
fraction = df["percent_inhibition_value"] / 100.0
df["pseudo_ic50_M_n1"] = np.where(
    valid_single_point,
    df["test_concentration_M"] * np.power((1.0 - fraction) / fraction, 1.0 / HILL_N_ASSUMED),
    np.nan,
)
df["pseudo_pic50_n1"] = np.where(df["pseudo_ic50_M_n1"] > 0, -np.log10(df["pseudo_ic50_M_n1"]), np.nan)
df["hill_assumed"] = np.where(valid_single_point, HILL_N_ASSUMED, np.nan)

bound_base = (
    df["is_percent_inhibition"]
    & df["test_concentration_M"].notna()
    & (df["test_concentration_M"] > 0)
    & df["percent_inhibition_value"].notna()
)
df["p_ic50_bound_value"] = np.where(bound_base, -np.log10(df["test_concentration_M"]), np.nan)
df["p_ic50_bound_relation"] = None
df.loc[bound_base & (df["percent_inhibition_value"] > 50), "p_ic50_bound_relation"] = ">"
df.loc[bound_base & (df["percent_inhibition_value"] < 50), "p_ic50_bound_relation"] = "<"
df.loc[bound_base & (df["percent_inhibition_value"] == 50), "p_ic50_bound_relation"] = "="

df["pseudo_ic50_method"] = None
df.loc[df["is_percent_inhibition"], "pseudo_ic50_method"] = "not_computed_missing_concentration_or_percent"
df.loc[valid_single_point, "pseudo_ic50_method"] = "single_point_percent_n1"

df["p_ic50_bound_method"] = None
df.loc[df["is_percent_inhibition"], "p_ic50_bound_method"] = "no_bound_missing_concentration_or_percent"
df.loc[bound_base, "p_ic50_bound_method"] = "single_point_bound_from_test_concentration"

df["p_ic50_modeling"] = df["p_ic50_ki"]
df["p_ic50_modeling_relation"] = df["p_ic50_ki_relation"]
df["p_ic50_modeling_method"] = np.where(df["p_ic50_ki"].notna(), "measured_ic50_or_ki", None)

pseudo_fill = df["p_ic50_modeling"].isna() & df["pseudo_pic50_n1"].notna()
df.loc[pseudo_fill, "p_ic50_modeling"] = df.loc[pseudo_fill, "pseudo_pic50_n1"]
df.loc[pseudo_fill, "p_ic50_modeling_relation"] = "="
df.loc[pseudo_fill, "p_ic50_modeling_method"] = "pseudo_from_percent_single_point_n1"

bound_fill = (
    df["p_ic50_modeling"].isna()
    & df["p_ic50_bound_value"].notna()
    & df["p_ic50_bound_relation"].notna()
)
df.loc[bound_fill, "p_ic50_modeling"] = df.loc[bound_fill, "p_ic50_bound_value"]
df.loc[bound_fill, "p_ic50_modeling_relation"] = df.loc[bound_fill, "p_ic50_bound_relation"]
df.loc[bound_fill, "p_ic50_modeling_method"] = "bound_from_percent_single_point"

validate_smiles, smiles_method = smiles_validator()
df["smiles_missing"] = df["canonical_smiles"].isna()
df["smiles_is_valid"] = df["canonical_smiles"].map(validate_smiles)
df["smiles_validation_method"] = smiles_method

has_measure = df["standard_type"].notna() | df["standard_value"].notna() | df["standard_units"].notna()
core_df = df[df["assay_chembl_id"].notna() & df["molecule_chembl_id"].notna() & has_measure].copy()
filtered_df = core_df[(~core_df["smiles_missing"]) & core_df["smiles_is_valid"]].copy()

filtered_std = filtered_df["standard_type"].fillna("").str.upper()
filtered_df["priority_has_modeled_pic50"] = filtered_df["p_ic50_modeling"].notna().astype(int)
filtered_df["priority_score"] = (
    8 * filtered_std.isin({"IC50", "KI"}).astype(int)
    + 4
    * filtered_df["assay_organism"]
    .fillna("")
    .str.contains(r"homo sapiens|human", case=False, regex=True)
    .astype(int)
    + 2 * filtered_df["relation_preserved"].eq("=").astype(int)
    + filtered_df["standard_value_nM"].notna().astype(int)
    + filtered_df["priority_has_modeled_pic50"]
)

max_priority = filtered_df.groupby(GROUP_COLS)["priority_score"].transform("max")
selected_df = filtered_df[filtered_df["priority_score"] == max_priority].copy()

numeric_cols = [
    "standard_value_num",
    "standard_value_nM",
    "standard_value_M",
    "p_activity_bound",
    "p_ic50_ki",
    "pchembl_value_num",
    "test_concentration_M",
    "test_concentration_raw_value",
    "percent_inhibition_value",
    "pseudo_ic50_M_n1",
    "pseudo_pic50_n1",
    "p_ic50_bound_value",
    "p_ic50_modeling",
    "hill_assumed",
]

selected_sorted = selected_df.sort_values(
    by=GROUP_COLS + ["priority_score", "p_ic50_modeling"],
    ascending=[True, True, False, False],
    kind="stable",
)
representative = selected_sorted.groupby(GROUP_COLS, as_index=False).agg(first_non_null)
medians = selected_df.groupby(GROUP_COLS, as_index=False)[numeric_cols].median(numeric_only=True)
medians = medians.rename(columns={c: f"{c}_median" for c in numeric_cols})
count_selected = selected_df.groupby(GROUP_COLS, as_index=False).size().rename(
    columns={"size": "n_selected_measurements"}
)
count_total = filtered_df.groupby(GROUP_COLS, as_index=False).size().rename(
    columns={"size": "n_total_measurements_in_group"}
)

cleaned_df = representative.merge(medians, on=GROUP_COLS, how="left")
cleaned_df = cleaned_df.merge(count_selected, on=GROUP_COLS, how="left")
cleaned_df = cleaned_df.merge(count_total, on=GROUP_COLS, how="left")

for col in numeric_cols:
    cleaned_df[col] = cleaned_df[f"{col}_median"].combine_first(cleaned_df[col])

cleaned_df = cleaned_df.sort_values(GROUP_COLS, kind="stable").reset_index(drop=True)
percent_df = cleaned_df[cleaned_df["is_percent_inhibition"]].copy()

cleaned_df.to_csv(CLEAN_OUTPUT_PATH, index=False)
percent_df.to_csv(PERCENT_OUTPUT_PATH, index=False)

counts = {
    "raw_rows": int(len(df_raw)),
    "core_rows": int(len(core_df)),
    "removed_missing_smiles": int(core_df["smiles_missing"].sum()),
    "removed_invalid_smiles": int((~core_df["smiles_missing"] & ~core_df["smiles_is_valid"]).sum()),
    "rows_after_structure_filter": int(len(filtered_df)),
    "rows_converted_nM": int(filtered_df["standard_value_nM"].notna().sum()),
    "rows_converted_pIC50_pKi": int(filtered_df["p_ic50_ki"].notna().sum()),
    "percent_inhibition_rows": int(filtered_df["is_percent_inhibition"].sum()),
    "percent_rows_with_parsed_test_concentration": int(
        (filtered_df["is_percent_inhibition"] & filtered_df["test_concentration_M"].notna()).sum()
    ),
    "percent_rows_with_pseudo_pic50": int(
        (filtered_df["is_percent_inhibition"] & filtered_df["pseudo_pic50_n1"].notna()).sum()
    ),
    "percent_rows_with_bound_pic50": int(
        (
            filtered_df["is_percent_inhibition"]
            & filtered_df["p_ic50_bound_value"].notna()
            & filtered_df["p_ic50_bound_relation"].notna()
        ).sum()
    ),
    "rows_with_any_modeling_pic50": int(filtered_df["p_ic50_modeling"].notna().sum()),
    "censored_rows": int(filtered_df["relation_preserved"].isin({">", "<", ">=", "<="}).sum()),
    "rows_after_duplicate_resolution": int(len(cleaned_df)),
    "duplicate_rows_collapsed": int(len(filtered_df) - len(cleaned_df)),
    "smiles_validation_method": smiles_method,
}

log_lines = [
    "Data cleaning and standardization log",
    "=====================================",
    "",
    "Rules implemented:",
    "1. Standardized concentration units to nM where units were in {M, mM, uM/um, nM, pM, fM}.",
    "2. Converted IC50/Ki rows to pIC50/pKi using -log10(M) when concentration values were available.",
    "3. Preserved inequality relations and labeled censoring; p-scale relations are inverted from concentration scale.",
    "4. For percent inhibition rows, parsed test concentration from assay text and computed pseudo-pIC50 (Hill=1) when feasible.",
    "5. For percent rows without pseudo estimates, generated inequality-style pIC50 bounds from test concentration when possible.",
    "6. Removed rows with missing/invalid SMILES from the cleaned output, while retaining flags for auditability.",
    "7. Resolved duplicates per compound-assay by prioritized selection (IC50/Ki, human assay, uncensored, numeric concentration, modeled pIC50) and median aggregation for numeric values.",
    "",
    "Counts:",
    f"- N raw rows: {counts['raw_rows']}",
    f"- Core rows (assay + molecule + measure present): {counts['core_rows']}",
    f"- Removed for missing SMILES: {counts['removed_missing_smiles']}",
    f"- Removed for invalid SMILES: {counts['removed_invalid_smiles']}",
    f"- Rows after structure filter: {counts['rows_after_structure_filter']}",
    f"- Converted to nM: {counts['rows_converted_nM']}",
    f"- Converted to pIC50/pKi (measured IC50/Ki): {counts['rows_converted_pIC50_pKi']}",
    f"- Percent inhibition rows flagged: {counts['percent_inhibition_rows']}",
    f"- Percent rows with parsed test concentration: {counts['percent_rows_with_parsed_test_concentration']}",
    f"- Percent rows with pseudo-pIC50 (single-point, n=1): {counts['percent_rows_with_pseudo_pic50']}",
    f"- Percent rows with pIC50 bound: {counts['percent_rows_with_bound_pic50']}",
    f"- Rows with any modeling pIC50 (measured or inferred): {counts['rows_with_any_modeling_pic50']}",
    f"- Censored rows preserved: {counts['censored_rows']}",
    f"- Rows after duplicate resolution: {counts['rows_after_duplicate_resolution']}",
    f"- Duplicate rows collapsed: {counts['duplicate_rows_collapsed']}",
    f"- SMILES validation method: {counts['smiles_validation_method']}",
    "",
    "Outputs:",
    "- data/herg_postprocessed.csv",
    "- data/herg_percent_inhibition.csv",
    "- filtering.log",
]
LOG_PATH.write_text("\n".join(log_lines) + "\n", encoding="utf-8")

print("Created:", CLEAN_OUTPUT_PATH)
print("Created:", PERCENT_OUTPUT_PATH)
print("Created:", LOG_PATH)
for key, value in counts.items():
    print(f"{key}: {value}")


Created: data\herg_postprocessed.csv
Created: data\herg_percent_inhibition.csv
Created: filtering.log
raw_rows: 2861
core_rows: 2861
removed_missing_smiles: 1
removed_invalid_smiles: 0
rows_after_structure_filter: 2860
rows_converted_nM: 2149
rows_converted_pIC50_pKi: 1687
percent_inhibition_rows: 439
percent_rows_with_parsed_test_concentration: 249
percent_rows_with_pseudo_pic50: 196
percent_rows_with_bound_pic50: 224
rows_with_any_modeling_pic50: 1911
censored_rows: 650
rows_after_duplicate_resolution: 2661
duplicate_rows_collapsed: 199
smiles_validation_method: regex_fallback_no_rdkit
